[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/03-index-configuration/defining_explicit_field_configs.ipynb)

# Defining Explicit Field Configs

Every index you've built so far let `TableIndexer` guess the right setup for each column, and attached any `CharacterMapping` or `AliasSet` through separate dictionaries passed alongside the DataFrame. That works, but it splits your schema across several places: the index type is inferred, the harmonization rules live in dictionaries defined elsewhere, and nothing about the code makes clear how a field is actually going to behave.

`TableFieldConfig` lets you declare everything about a single column,  its `index_type`, its `character_mapping`, and its `aliases`,  in one object.
The `TableConfig` lets you to build and assemble your schema. 

In this notebook you will:

1. See a case where automatic type inference produces the wrong result
2. Review every `IndexType` and what each one is actually for
3. Build a `TableFieldConfig` that bundles an `index_type` *and* a `character_mapping` *and* `aliases` together, for the same field
4. Assemble a full schema and compile an index from it
5. Know when explicit configuration is worth the extra code, and when inference is fine

In [2]:
# !pip install mbox

## 1. Where inference quietly goes wrong

Here's a small product catalog, including a UPC barcode column.

In [3]:
import pandas as pd
from mbox.indexing import TableIndexer

df = pd.DataFrame({
    "product_id": ["B88-EXT", "A12-PWR", "C99-SNS", "D45-REL"],
    "product_name": [
        "Extended Battery Pack",
        "Portable Power Bank",
        "Motion Sensor Camera",
        "Smart Relay Switch"
    ],
    "upc_code": [12345678, 98765432, 45612378, 78912345],  # stored as int64
    "quantity_in_stock": [140, 76, 212, 58],
    "unit_price": [24.99, 39.50, 59.00, 18.75],
    "warehouse_notes": ["Reorder soon", "Stable", "New shipment incoming", "Discontinuing Q3"]
})

print(df.dtypes)
df

product_id               str
product_name             str
upc_code               int64
quantity_in_stock      int64
unit_price           float64
warehouse_notes          str
dtype: object


,product_id,product_name,upc_code,quantity_in_stock,unit_price,warehouse_notes
0,B88-EXT,Extended Battery Pack,12345678,140,24.99,Reorder soon
1,A12-PWR,Portable Power Bank,98765432,76,39.50,Stable
2,C99-SNS,Motion Sensor Camera,45612378,212,59.00,New shipment incoming
3,D45-REL,Smart Relay Switch,78912345,58,18.75,Discontinuing Q3


`upc_code` and `quantity_in_stock` are both `int64`,  but they mean completely different things. A quantity is a real number where magnitude matters: 140 units is meaningfully more than 76. A UPC barcode is an identifier that happens to be made of digits,  there's no meaningful sense in which one barcode is numerically "close" to another, and if any of these codes legitimately started with a `0`, storing them as `int64` would silently drop it, the same way a postal code would.

Automatic inference has no way to know this. Every integer-dtype column gets `IndexType.INTEGER`, full stop,  whether it's a quantity or an identifier. That's exactly the kind of silent, wrong-by-default behavior explicit configuration is for.

## 2. The `IndexType` reference

Every field in an index is assigned one of these strategies, whether inferred or set explicitly:

| `IndexType` | Supported dtypes | Best used for |
|---|---|---|
| `PHRASE` | String | Full text, multi-word strings, descriptions |
| `TERM` | String | Single-word tokens,  categories, single-word names |
| `IDENT` | String | Identifiers with minor variation tolerance,  SKUs, product codes, barcodes |
| `INTEGER` | Integer | Whole numbers where magnitude is meaningful,  quantities, years |
| `DOUBLE` | Float / numeric | Floating-point values,  prices, coordinates, ratings |
| `NON_SEARCHABLE` | Any | Payload fields returned in results but never searched against |

`upc_code` belongs in `IDENT`, not `INTEGER`,  it's an identifier, not a quantity.

## 3. Fixing the dtype before fixing the config

Before declaring `IndexType.IDENT`, the column itself needs to actually be a string,  `IndexType` can't recover a digit that pandas already dropped. This step has nothing to do with M|BOX specifically; it's just good practice for any identifier-like column.

In [4]:
df["upc_code"] = df["upc_code"].astype(str).str.zfill(8)
df["upc_code"]

0    12345678
1    98765432
2    45612378
3    78912345
Name: upc_code, dtype: str

## 4. Bundling harmonization directly into a field's config

Recall from `02-data-harmonization/combining_mappings_and_aliases_in_one_index.ipynb` that `product_id` has two independent problems: OCR-style scanning noise (`"B88-EXT"` misread as `"8B8-3XT"`) and legacy product codes from an older system (`"LGCY-441"`). We solved both there by passing dictionaries into `TableIndexer.create_index()`.

`TableFieldConfig` lets you attach the exact same `CharacterMapping` and `AliasSet` directly on the field's own configuration object instead,  so everything about how `product_id` behaves lives in one place.

In [5]:
from mbox.mapping import CharacterMapping
from mbox.aliases import AliasSet
from mbox.config import TableConfig, TableFieldConfig, IndexType

# Same custom mapping as before: OCR-style digit/letter confusion, no umlaut handling needed
product_id_mapping = CharacterMapping(
    name="product_id_ocr_cleaner",
    mapped_characters="A-Z0-9-",
    map_upper=True,
    deaccentuate=True,
    expand_umlauts=False,
    numbers_as_characters=True
)

# Same legacy code aliases as before
legacy_code_aliases = AliasSet(name="legacy_product_codes")
legacy_code_aliases.add(word="B88-EXT", alias="LGCY-441", penalty=0)
legacy_code_aliases.add(word="A12-PWR", alias="LGCY-198", penalty=0)

# Both attached directly on the same TableFieldConfig
product_id_field = TableFieldConfig(
    column="product_id",
    index_type=IndexType.IDENT,
    character_mapping=product_id_mapping,
    aliases=legacy_code_aliases
)

`product_id_field` now fully describes how this column behaves: its type, its spelling normalization, and its known synonyms, all declared together. Nothing about `product_id`'s behavior lives outside this one object.

## 5. Declaring the rest of the fields

Not every field needs harmonization,  `character_mapping` and `aliases` are optional and default to `None`. Let's declare the remaining columns.

In [6]:
product_name_field = TableFieldConfig(
    column="product_name",
    index_type=IndexType.PHRASE
)

upc_code_field = TableFieldConfig(
    column="upc_code",
    index_type=IndexType.IDENT   # explicit override -- not a quantity
)

quantity_field = TableFieldConfig(
    column="quantity_in_stock",
    index_type=IndexType.INTEGER   # this one really is a quantity
)

unit_price_field = TableFieldConfig(
    column="unit_price",
    index_type=IndexType.DOUBLE
)

warehouse_notes_field = TableFieldConfig(
    column="warehouse_notes",
    index_type=IndexType.NON_SEARCHABLE
)

## 6. Assembling the schema and building the index

Gather every `TableFieldConfig` into a `TableConfig`, and pass it to `TableIndexer.create_index()` via `config_overrides` instead of `index_columns`.

In [7]:
schema = TableConfig(fields=[
    product_id_field,
    product_name_field,
    upc_code_field,
    quantity_field,
    unit_price_field,
    warehouse_notes_field
])

index = TableIndexer.create_index(df=df, config_overrides=schema, tmp_dir="tmp_index")
index

config_overrides is given, therefore all information from index_columns, index_types, alias_sets and character_mappings is ignored.


Confirm the schema applied exactly as declared:

In [8]:
index.describe()

,Field,Index Type,Unique Value Count,Index size [bytes]
0,product_id,IndexType.IDENT,6,10436
1,product_name,IndexType.PHRASE,4,25248
2,upc_code,IndexType.IDENT,4,10500
3,quantity_in_stock,IndexType.INTEGER,4,124
4,unit_price,IndexType.DOUBLE,4,156
5,warehouse_notes,IndexType.NON_SEARCHABLE,4,0


`upc_code` should show `IndexType.IDENT`, not `IndexType.INTEGER`. `warehouse_notes` should show `IndexType.NON_SEARCHABLE` with no index size. And `product_id` should carry the harmonization we attached,  let's confirm that part actually works by running the same two problem queries from the data-harmonization notebook.

In [9]:
ocr_query = index.match(product_id="8B8-3XT", include_field_scores=True)
legacy_query = index.match(product_id="LGCY-441", include_field_scores=True)

print("OCR-garbled query:")
display(ocr_query)

print("\nLegacy code query:")
display(legacy_query)

OCR-garbled query:


,query_row,index_row,product_id_candidate,product_name_candidate,upc_code_candidate,quantity_in_stock_candidate,unit_price_candidate,warehouse_notes_candidate,overall_score,product_id_score
0,0,0,B88-EXT,Extended Battery Pack,12345678,140,24.99,Reorder soon,100,100



Legacy code query:


,query_row,index_row,product_id_candidate,product_name_candidate,upc_code_candidate,quantity_in_stock_candidate,unit_price_candidate,warehouse_notes_candidate,overall_score,product_id_score
0,0,0,B88-EXT,Extended Battery Pack,12345678,140,24.99,Reorder soon,100,100


Both should resolve to `"B88-EXT"` with strong scores,  identical behavior to the dict-based approach from `02-data-harmonization/`, just declared through `TableFieldConfig` this time instead of separate `character_mappings=` and `alias_sets=` dictionaries. The two approaches produce the same result; explicit field configs just make the schema itself the single source of truth.

## 7. When explicit configuration is worth it

Writing out a `TableFieldConfig` for every column, with harmonization attached inline, is more code than the dictionary-based shortcuts. It's worth the extra lines when:

- **An identifier could be mistaken for a quantity**,  barcodes, account numbers, or codes stored as numbers, where `INTEGER`'s numeric-distance semantics would be wrong
- **A field's harmonization rules are complex enough that they deserve to live with the field itself**,  rather than in a separate dictionary someone has to cross-reference to understand what `product_id` actually does
- **You're building something others will maintain**,  the schema documents intent directly in code
- **You need the schema to be portable**,  explicit configs pair naturally with version-controlled JSON, covered next

For quick, one-off exploration, the dictionary shortcuts from `02-data-harmonization/` are fine and often faster to write. Reach for `TableFieldConfig` once a schema is something you expect to reuse, hand off, or maintain over time.

## Next steps

- **`managing_schemas_with_tableconfig.ipynb`**,  add, update, remove, and validate fields on a `TableConfig` dynamically, rather than building the whole list up front
- **`schema_json_serialization_and_reuse.ipynb`**,  save a schema like this one to JSON and load it in another service or pipeline run

*M|BOX is currently in `beta`. Breaking changes may occur in minor releases until version `1.0.0`.*